Preprocessing function

In [1]:
import re

def preprocess_complaint(text):

    if not isinstance(text, str):
        return ""

    # lowercase
    text = text.lower()

    # remove masked dates like xx/xx/xxxx
    text = re.sub(r'\b[x]{2}/[x]{2}/[x]{4}\b', ' ', text)

    # remove masked numbers like xxxx1234 or xxxx
    text = re.sub(r'\b[x]{2,}\d*\b', ' ', text)

    # normalize whitespace
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

Mount Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Load data

In [3]:
import pandas as pd

train_df = pd.read_parquet('/content/drive/MyDrive/CFPB_Project/processed/train_complaints.parquet')
test_df  = pd.read_parquet('/content/drive/MyDrive/CFPB_Project/processed/test_complaints.parquet')

In [4]:
print(train_df.shape)
print(test_df.shape)

(812720, 4)
(203180, 4)


In [5]:
train_df.head()

,Complaint ID,Consumer complaint narrative,Department,Priority
0,7251882,This is regarding the Texas B-On-Time Student ...,Student loan,high_priority
1,16800276,"On top of other discrepancies, XXXX XXXX accou...",Credit reporting,standard
2,2292018,There was fraudulent activity on my account wh...,Bank accounts,high_priority
3,7835057,I have consistently ensured my payments are ma...,Card services,standard
4,4313529,I tried to send them a letter last XX/XX/2020....,Mortgage,standard


Define features and targets

In [6]:
X_train = train_df["Consumer complaint narrative"]
X_test  = test_df["Consumer complaint narrative"]

y_train_dept = train_df["Department"]
y_test_dept  = test_df["Department"]

y_train_priority = train_df["Priority"]
y_test_priority  = test_df["Priority"]

Preprocess text

In [7]:
X_train_clean = [preprocess_complaint(x) for x in X_train]
X_test_clean  = [preprocess_complaint(x) for x in X_test]

Encode Labels

In [8]:
from sklearn.preprocessing import LabelEncoder

le_dept = LabelEncoder()
y_train_dept_enc = le_dept.fit_transform(y_train_dept)
y_test_dept_enc  = le_dept.transform(y_test_dept)

le_priority = LabelEncoder()
y_train_priority_enc = le_priority.fit_transform(y_train_priority)
y_test_priority_enc  = le_priority.transform(y_test_priority)

Tokenization

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=256
    )

train_encodings = tokenize(X_train_clean)
test_encodings  = tokenize(X_test_clean)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Create Dataset

In [10]:
import torch

class ComplaintDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

Build datasets - Department

In [11]:
train_dataset = ComplaintDataset(train_encodings, y_train_dept_enc)
test_dataset  = ComplaintDataset(test_encodings, y_test_dept_enc)

Load Model

In [12]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(le_dept.classes_)
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training Setup

In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",  # <-- important fix
    save_strategy="epoch",
    load_best_model_at_end=True
)

Metrics

In [14]:
from sklearn.metrics import f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}

Train

In [15]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1
1,0.347915,0.346161,0.804394


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Macro F1
1,0.347915,0.346161,0.804394
2,0.322052,0.331365,0.813853
3,0.253132,0.351469,0.814253


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=152385, training_loss=0.3231286872327807, metrics={'train_runtime': 13285.4591, 'train_samples_per_second': 183.521, 'train_steps_per_second': 11.47, 'total_flos': 1.6150851576262656e+17, 'train_loss': 0.3231286872327807, 'epoch': 3.0})

Evaluate

In [16]:
trainer.evaluate()

{'eval_loss': 0.33132290840148926,
 'eval_macro_f1': 0.8137948592532697,
 'eval_runtime': 349.1767,
 'eval_samples_per_second': 581.883,
 'eval_steps_per_second': 36.368,
 'epoch': 3.0}

Save model

In [17]:
trainer.save_model("/content/drive/MyDrive/final_model")
tokenizer.save_pretrained("/content/drive/MyDrive/final_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/final_model/tokenizer_config.json',
 '/content/drive/MyDrive/final_model/tokenizer.json')

Check label mapping

In [18]:
id2label = dict(enumerate(le_dept.classes_))
label2id = {v: k for k, v in id2label.items()}

In [19]:
print(list(le_dept.classes_))
print(dict(enumerate(le_dept.classes_)))

['Bank accounts', 'Card services', 'Consumer loans', 'Credit reporting', 'Debt collection', 'Money transfer services', 'Mortgage', 'Payday / personal loans', 'Student loan']
{0: 'Bank accounts', 1: 'Card services', 2: 'Consumer loans', 3: 'Credit reporting', 4: 'Debt collection', 5: 'Money transfer services', 6: 'Mortgage', 7: 'Payday / personal loans', 8: 'Student loan'}
